# Step 1
Establish one shared baseline dataset pipeline for Section 3.1 Data Description

## Success criterion for Step 1
Orange3 and Jupyter must produce the same row count, same column count, and the same validation numbers before moving to the next report section.

In [6]:
# Step 1 implementation: shared baseline data pipeline + auditable checks
from pathlib import Path
import json

# Load train.csv and items.csv with separator |
train_set = pd.read_csv("data/train.csv", sep="|")
items_set = pd.read_csv("data/items.csv", sep="|")

# Left join on pid to enrich train data with item attributes
train_item_set = train_set.merge(items_set, on="pid", how="left")

# Validation checks
row_count = int(train_item_set.shape[0])
column_count = int(train_item_set.shape[1])
missing_values = train_item_set.isnull().sum()

# Strict mutual exclusivity: exactly one of click/basket/order must be 1
action_sum = train_item_set[["click", "basket", "order"]].sum(axis=1)
mutual_exclusive_exact = bool((action_sum == 1).all())

print("Row count:", row_count)
print("Column count:", column_count)
print("Missing values per column:")
print(missing_values)
print("Click/basket/order mutual exclusivity (exactly one):", mutual_exclusive_exact)

# Export canonical Step 1 baseline dataset
processed_dir = Path("data") / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
baseline_path = processed_dir / "step1_merged_baseline.csv"
train_item_set.to_csv(baseline_path, index=False)

# Export auditable validation artifacts for Orange3/Jupyter parity checks
missing_path = processed_dir / "step1_missing_values.csv"
missing_values.rename("missing_count").to_csv(missing_path)

summary = {
    "row_count": row_count,
    "column_count": column_count,
    "mutual_exclusive_exact": mutual_exclusive_exact,
    "dataset_file": str(baseline_path),
    "missing_values_file": str(missing_path)
}
summary_path = processed_dir / "step1_validation_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:", baseline_path)
print("Saved:", missing_path)
print("Saved:", summary_path)

Row count: 2756003
Column count: 21
Missing values per column:
lineID                   0
day                      0
pid                      0
adFlag                   0
availability             0
competitorPrice     100687
click                    0
basket                   0
order                    0
price                    0
revenue                  0
manufacturer             0
group                    0
content                  0
unit                     0
pharmForm           194124
genericProduct           0
salesIndex               0
category             87394
campaignIndex      2287968
rrp                      0
dtype: int64
Click/basket/order mutual exclusivity (exactly one): True
Saved: data/processed/step1_merged_baseline.csv
Saved: data/processed/step1_missing_values.csv
Saved: data/processed/step1_validation_summary.json
